# 01: Veri Seti Tanıtımı ve Ön İşleme (Kişi 1)

**Proje:** Framingham Kalp Çalışması verisi ile inme (stroke) tahmini

**Bu notebook'ta yapılanlar:**
1. Veri setinin yüklenmesi ve tanıtımı
2. Problemin tanımlanması: hedef değişken, analize alınan kişiler
3. Veri sızıntısına yol açan sütunların çıkarılması
4. Eksik ve tutarsız verilerin tespiti
5. Anormal (aykırı) değerlerin incelenmesi
6. Eğitim/test ayrımı ve eksik verilerin doldurulması
7. Temiz verinin kaydedilmesi (`train.csv`, `test.csv`)

**Veri seti:** https://www.kaggle.com/datasets/shreyjain601/framingham-heart-study
(NHLBI Framingham Heart Study eğitim amaçlı boylamsal veri seti)

## 1. Kütüphaneler ve sabitler

In [ ]:
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42   # tüm ekip aynı değeri kullanır
TEST_ORANI = 0.20   # %80 eğitim, %20 test
HEDEF = "STROKE"

pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")

## 2. Veri setinin yüklenmesi

In [ ]:
import kagglehub
path = kagglehub.dataset_download("shreyjain601/framingham-heart-study")
csv_yolu = glob.glob(os.path.join(path, "**", "*.csv"), recursive=True)[0]
df_ham = pd.read_csv(csv_yolu)

print("Dosya:", os.path.basename(csv_yolu))
print("Boyut (satır, sütun):", df_ham.shape)
df_ham.head()

## 3. Veri setinin tanıtımı

Framingham Kalp Çalışması 1948'de ABD'nin Framingham kasabasında başlamış, kalp ve damar hastalıklarının risk faktörlerini inceleyen uzun süreli bir çalışmadır. Bu veri seti, 4.434 katılımcının 3 farklı muayene dönemindeki (PERIOD = 1, 2, 3) ölçümlerini ve yaklaşık 24 yıllık takip süresinde yaşadıkları olayları (inme, kalp krizi, ölüm vb.) içerir.

**Veri setindeki sütun grupları:**

| Grup | Sütunlar | Açıklama |
|---|---|---|
| Kimlik / zaman | `RANDID`, `PERIOD`, `TIME` | Kişi numarası, muayene dönemi, başlangıçtan bu yana geçen gün |
| Demografik | `SEX`, `AGE`, `educ` | Cinsiyet (1 = erkek, 2 = kadın), yaş, eğitim düzeyi (1-4) |
| Klinik ölçümler | `TOTCHOL`, `SYSBP`, `DIABP`, `BMI`, `HEARTRTE`, `GLUCOSE`, `HDLC`, `LDLC` | Kolesterol, büyük/küçük tansiyon, vücut kitle indeksi, nabız, glikoz |
| Yaşam tarzı / ilaç | `CURSMOKE`, `CIGPDAY`, `BPMEDS` | Sigara içme, günlük sigara sayısı, tansiyon ilacı |
| Muayene anındaki hastalıklar | `DIABETES`, `PREVCHD`, `PREVAP`, `PREVMI`, `PREVSTRK`, `PREVHYP` | Diyabet, koroner kalp hastalığı, anjina, kalp krizi, inme, hipertansiyon öyküsü |
| Takip sonuçları | `DEATH`, `ANGINA`, `HOSPMI`, `MI_FCHD`, `ANYCHD`, `STROKE`, `CVD`, `HYPERTEN` | Takip süresince olay yaşandı mı (0/1) |
| Olay zamanları | `TIMEAP`, `TIMEMI`, `TIMEMIFC`, `TIMECHD`, `TIMESTRK`, `TIMECVD`, `TIMEDTH`, `TIMEHYP` | Olaya kadar geçen gün sayısı |

In [ ]:
df_ham.info()

In [ ]:
print("Benzersiz katılımcı sayısı:", df_ham["RANDID"].nunique())
print("\nMuayene dönemlerine göre satır sayısı:")
print(df_ham["PERIOD"].value_counts().sort_index())

## 4. Problemin tanımlanması

**Hedef değişken:** `STROKE` (takip süresince inme geçirdi mi? 0 = hayır, 1 = evet)

Veri boylamsal olduğu için aynı kişi 3 satıra kadar tekrar edebilir. Bunu olduğu gibi kullanırsak aynı kişinin bir satırı eğitim, diğer satırı test verisine düşer ve model sonuçları gerçekte olduğundan iyi görünür. Bu yüzden:

1. **Sadece ilk muayene (PERIOD = 1) alınır.** Her kişi tek satırla temsil edilir ve tahmin, takibin başındaki bilgilerle yapılır.
2. **Muayene anında zaten inme geçirmiş kişiler (PREVSTRK = 1) çıkarılır.** Amaç yeni bir inmeyi önceden tahmin etmektir.

In [ ]:
df = df_ham[df_ham["PERIOD"] == 1].copy()
print("PERIOD = 1 sonrası:", df.shape)

onceki_inme = (df["PREVSTRK"] == 1).sum()
df = df[df["PREVSTRK"] == 0].copy()
print(f"Daha önce inme geçirmiş {onceki_inme} kişi çıkarıldı. Kalan: {df.shape}")

## 5. Veri sızıntısına yol açan sütunların çıkarılması

Bazı sütunlar ancak takip süresi boyunca ortaya çıkan bilgileri içerir. Gerçek hayatta tahmin anında bu bilgiler bilinemez. Model bunları görürse "geleceği bilerek" tahmin yapar ve sonuçlar yapay olarak yüksek çıkar (veri sızıntısı / data leakage).

| Çıkarılan sütunlar | Neden |
|---|---|
| `RANDID` | Kimlik numarası, bilgi taşımaz |
| `PERIOD`, `TIME` | Tüm satırlarda sabit (ilk muayene) |
| `PREVSTRK` | Filtrelemeden sonra hep 0 |
| `TIMESTRK` ve diğer tüm `TIME...` sütunları | Olaya kadar geçen süre: hedefi doğrudan ele verir |
| `DEATH`, `ANGINA`, `HOSPMI`, `MI_FCHD`, `ANYCHD`, `CVD`, `HYPERTEN` | Takip süresince oluşan diğer sonuçlar |

In [ ]:
sizinti = ["RANDID", "PERIOD", "TIME", "PREVSTRK",
           "DEATH", "ANGINA", "HOSPMI", "MI_FCHD", "ANYCHD", "CVD", "HYPERTEN"]
sizinti += [c for c in df.columns if c.startswith("TIME")]
sizinti = sorted(set(c for c in sizinti if c in df.columns))

df = df.drop(columns=sizinti)
print("Çıkarılan sütunlar:", sizinti)
print("Kalan boyut:", df.shape)
print("Kalan sütunlar:", list(df.columns))

## 6. Hedef değişkenin dağılımı

In [ ]:
dagilim = df[HEDEF].value_counts().sort_index()
oran = df[HEDEF].value_counts(normalize=True).sort_index() * 100
print(pd.DataFrame({"kişi sayısı": dagilim, "yüzde": oran.round(2)}))

ax = dagilim.plot(kind="bar", color=["#4C72B0", "#C44E52"])
ax.set_xticklabels(["İnme yok (0)", "İnme var (1)"], rotation=0)
ax.set_title("Hedef değişken dağılımı (STROKE)")
ax.set_ylabel("Kişi sayısı")
plt.show()

Sınıflar dengesizdir: inme geçiren kişiler azınlıktadır. Bu nedenle model değerlendirmesinde accuracy yerine recall, F1 ve ROC-AUC kullanılacak, dengeleme (SMOTE) ise sadece eğitim verisine uygulanacaktır (02 numaralı notebook).

## 7. Eksik verilerin tespiti

In [ ]:
eksik = df.isnull().sum()
eksik_tablo = pd.DataFrame({"eksik sayısı": eksik, "yüzde": (eksik / len(df) * 100).round(2)})
eksik_tablo = eksik_tablo[eksik_tablo["eksik sayısı"] > 0].sort_values("eksik sayısı", ascending=False)
print(eksik_tablo)

eksik_tablo["yüzde"].plot(kind="barh", color="#DD8452")
plt.title("Sütunlara göre eksik veri yüzdesi")
plt.xlabel("% eksik")
plt.gca().invert_yaxis()
plt.show()

**Yorum:** `HDLC` ve `LDLC` (iyi ve kötü kolesterol) yalnızca 3. muayene döneminde ölçülmüştür. İlk muayenede tamamen boş oldukları için doldurulamazlar ve çıkarılırlar. Diğer sütunlardaki eksikler ise az sayıdadır ve doldurulacaktır (bkz. Bölüm 10).

In [ ]:
cok_eksik = [c for c in df.columns if df[c].isnull().mean() > 0.5]
df = df.drop(columns=cok_eksik)
print("%50'den fazlası boş olduğu için çıkarılan sütunlar:", cok_eksik)
print("Kalan boyut:", df.shape)

## 8. Tekrar eden ve tutarsız kayıtlar

In [ ]:
print("Tekrar eden satır sayısı:", df.duplicated().sum())

# Tutarlılık kontrolleri
t1 = df[(df["CURSMOKE"] == 0) & (df["CIGPDAY"] > 0)]
t2 = df[(df["CURSMOKE"] == 1) & (df["CIGPDAY"] == 0)]
print("Sigara içmiyor ama günlük sigara > 0:", len(t1))
print("Sigara içiyor ama günlük sigara = 0:", len(t2))
print("Sigara içmeyen ama CIGPDAY'i boş olan:", (df["CIGPDAY"].isnull() & (df["CURSMOKE"] == 0)).sum())
print("Sigara içen ve CIGPDAY'i boş olan:", (df["CIGPDAY"].isnull() & (df["CURSMOKE"] == 1)).sum())
print("Küçük tansiyon >= büyük tansiyon:", (df["DIABP"] >= df["SYSBP"]).sum())

Sigara içip günlük sigara sayısı 0 olan kişiler mantıksal bir tutarsızlıktır. Bu değerler eksik veri (NaN) olarak işaretlenir ve aşağıda sigara içenlerin medyanı ile doldurulur.

In [ ]:
df.loc[(df["CURSMOKE"] == 1) & (df["CIGPDAY"] == 0), "CIGPDAY"] = np.nan
df.loc[(df["CURSMOKE"] == 0) & (df["CIGPDAY"].isnull()), "CIGPDAY"] = 0   # içmiyorsa sayı kesin 0
print("CIGPDAY eksik sayısı (düzeltme sonrası):", df["CIGPDAY"].isnull().sum())

## 9. Anormal (aykırı) değerlerin incelenmesi

In [ ]:
surekli = [c for c in ["AGE", "TOTCHOL", "SYSBP", "DIABP", "BMI", "HEARTRTE", "GLUCOSE", "CIGPDAY"] if c in df.columns]

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, c in zip(axes.ravel(), surekli):
    sns.boxplot(y=df[c], ax=ax, color="#8DA0CB")
    ax.set_title(c)
plt.suptitle("Sürekli değişkenlerde aykırı değerler (boxplot)")
plt.tight_layout()
plt.show()

# IQR yöntemine göre aykırı değer sayıları
satirlar = []
for c in surekli:
    q1, q3 = df[c].quantile([0.25, 0.75])
    iqr = q3 - q1
    alt, ust = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    satirlar.append([c, round(df[c].min(), 1), round(df[c].max(), 1), ((df[c] < alt) | (df[c] > ust)).sum()])
print(pd.DataFrame(satirlar, columns=["sütun", "min", "maks", "IQR aykırı sayısı"]).to_string(index=False))

**Klinik olarak mümkün olmayan değer kontrolü:** IQR yöntemi pek çok değeri aykırı olarak işaretler, ancak tıbbi verilerde yüksek tansiyon, yüksek glikoz ya da yüksek kolesterol gerçek ve **önemli** bilgilerdir (inme riskini doğrudan etkiler). Bu nedenle aykırı değerler silinmez. Yalnızca insan fizyolojisine göre imkânsız olan değerler (ölçüm/kayıt hatası) kaldırılır.

In [ ]:
klinik_sinirlar = {          # (en düşük, en yüksek) makul değer
    "SYSBP":    (70, 300),
    "DIABP":    (30, 200),
    "BMI":      (12, 70),
    "HEARTRTE": (30, 220),
    "GLUCOSE":  (30, 600),
    "TOTCHOL":  (80, 700),
}
hatali = pd.Series(False, index=df.index)
for c, (alt, ust) in klinik_sinirlar.items():
    if c in df.columns:
        maske = df[c].notnull() & ((df[c] < alt) | (df[c] > ust))
        print(f"{c}: {maske.sum()} imkânsız değer")
        hatali |= maske

tansiyon_hatasi = df["DIABP"] >= df["SYSBP"]
print(f"Küçük tansiyon >= büyük tansiyon: {tansiyon_hatasi.sum()} kayıt hatası")
hatali |= tansiyon_hatasi

df = df[~hatali].copy()
print("\nKlinik kontrol sonrası boyut:", df.shape)

## 10. Eğitim/test ayrımı ve eksik verilerin doldurulması

Eksik değerleri doldurmak için kullanılan istatistikler (medyan, mod) **yalnızca eğitim verisinden** hesaplanır ve test verisine aynen uygulanır. Böylece test verisindeki bilgiler eğitim sürecine sızmaz.

- **Ayırma yöntemi:** Hold-out, %80 eğitim / %20 test, hedefe göre katmanlı (stratified), `random_state = 42`
- **Sürekli değişkenler** (`TOTCHOL`, `BMI`, `HEARTRTE`, `GLUCOSE`): **medyan**. Aykırı değerlerden etkilenmez.
- **`CIGPDAY`**: sigara içenlerin medyanı (içmeyenler zaten 0 yapıldı)
- **Kategorik / ikili** (`educ`, `BPMEDS`): **mod** (en sık değer)

In [ ]:
X = df.drop(columns=[HEDEF])
y = df[HEDEF]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_ORANI, stratify=y, random_state=RANDOM_STATE)

print("Eğitim:", X_train.shape, "| İnme oranı: %.2f%%" % (y_train.mean() * 100))
print("Test:  ", X_test.shape, "| İnme oranı: %.2f%%" % (y_test.mean() * 100))

In [ ]:
X_train = X_train.copy(); X_test = X_test.copy()

medyan_sutunlar = [c for c in ["TOTCHOL", "BMI", "HEARTRTE", "GLUCOSE"] if c in X_train.columns]
mod_sutunlar    = [c for c in ["educ", "BPMEDS"] if c in X_train.columns]

doldurma = {}
for c in medyan_sutunlar:
    doldurma[c] = X_train[c].median()
for c in mod_sutunlar:
    doldurma[c] = X_train[c].mode()[0]
doldurma["CIGPDAY"] = X_train.loc[X_train["CURSMOKE"] == 1, "CIGPDAY"].median()

X_train = X_train.fillna(doldurma)
X_test  = X_test.fillna(doldurma)

print("Doldurma değerleri (eğitim verisinden):")
for k, v in doldurma.items():
    print(f"  {k}: {v}")
print("\nKalan eksik (eğitim):", X_train.isnull().sum().sum(), "| (test):", X_test.isnull().sum().sum())

## 11. Son durum ve kaydetme

In [ ]:
print("Son özellik sayısı:", X_train.shape[1])
print("Özellikler:", list(X_train.columns))
X_train.describe().T.round(2)

In [ ]:
train = X_train.assign(**{HEDEF: y_train})
test  = X_test.assign(**{HEDEF: y_test})
train.to_csv("train.csv", index=False)
test.to_csv("test.csv", index=False)
print("Kaydedildi: train.csv", train.shape, "| test.csv", test.shape)

In [ ]:
# Colab'da dosyaları bilgisayara indirmek için (ekiple paylaşın veya Drive'a koyun)
try:
    from google.colab import files
    files.download("train.csv")
    files.download("test.csv")
except ImportError:
    print("Colab dışında çalışıyor, dosyalar bulunduğunuz klasöre kaydedildi.")

## 12. Özet (rapor için)

- Ham veri: 11.627 satır, 39 sütun (4.434 katılımcının 3 muayene dönemi).
- Her katılımcının yalnızca ilk muayenesi alındı, muayene anında inme öyküsü olanlar çıkarıldı.
- Takip süresince oluşan bilgileri içeren sütunlar veri sızıntısını önlemek için çıkarıldı.
- İlk muayenede ölçülmemiş `HDLC` ve `LDLC` sütunları çıkarıldı.
- Sigara verisindeki mantıksal tutarsızlıklar düzeltildi.
- Aykırı değerler klinik olarak anlamlı olduğu için korundu, yalnızca fizyolojik olarak imkânsız değerler kaldırıldı.
- Veri %80/%20 katmanlı olarak ayrıldı, eksik değerler yalnızca eğitim verisinden hesaplanan medyan/mod ile dolduruldu.
- Çıktılar: `train.csv` ve `test.csv` (Kişi 2 bu dosyalarla encoding, ölçekleme ve SMOTE adımlarına devam eder).